## Bronze Layer

In [0]:
# Διάβασμα του πίνακα που δημιουργήθηκε στο Catalog
taxis_df = spark.read.parquet("/Volumes/workspace/default/yello_taxi_nyc/Yellow Taxi Data/")

# Εμφάνιση των πρώτων 5 γραμμών
display(taxis_df.limit(5))

In [0]:
# Διάβασμα του αρχείου καιρού (CSV)
weather_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("comment", "#") \
    .csv("/Volumes/workspace/default/yello_taxi_nyc/open-meteo-40.81N74.02W44m.csv") 

# Εμφάνιση των πρώτων 5 γραμμών του καιρού
display(weather_df.limit(5))

In [0]:
# 1. Δες τη δομή (Schema) των δεδομένων των ταξί
taxis_df.printSchema()

# 2. Μέτρησε το σύνολο των εγγραφών (Rows)
print(f"Συνολικές γραμμές στα δεδομένα ταξί: {taxis_df.count():,}")

In [0]:
# 1. Δες τη δομή (Schema) των δεδομένων των ταξί
weather_df.printSchema()

# 2. Μέτρησε το σύνολο των εγγραφών (Rows)
print(f"Συνολικές γραμμές στα δεδομένα καιρού: {weather_df.count():,}")

## Silver Layer

In [0]:
from pyspark.sql.functions import col, to_date

# 1. Εφαρμογή φίλτρων (Data Cleaning)
taxis_silver_df = taxis_df.filter(
    (col("tpep_pickup_datetime") >= "2026-01-01 00:00:00") & 
    (col("tpep_pickup_datetime") <= "2026-03-31 23:59:59") & 
    (col("trip_distance") > 0) & 
    (col("total_amount") > 0) &
    (col("passenger_count") > 0)
)

# 2. Δημιουργία στήλης 'pickup_date' (YYYY-MM-DD) για να κουμπώσει με τον καιρό
taxis_silver_df = taxis_silver_df.withColumn("pickup_date", to_date(col("tpep_pickup_datetime")))

# 3. Έλεγχος: Πόσες "βρώμικες" γραμμές πετάξαμε;
original_count = 11077206
cleaned_count = taxis_silver_df.count()
deleted_rows = original_count - cleaned_count

print(f"Καθαρές γραμμές: {cleaned_count:,}")
print(f"Γραμμές που διαγράφηκαν ως λάθος: {deleted_rows:,}")

In [0]:
from pyspark.sql.functions import col, to_date

# 1. Κρατάμε μόνο τις γραμμές που έχουν πραγματικές ημερομηνίες 
# (δηλαδή πετάμε τη γραμμή "time" και τη γραμμή με τις συντεταγμένες)
weather_silver_df = weather_df.filter(
    (col("latitude") != "time") & 
    (col("latitude") != "40.808434")
)

# 2. Επιλέγουμε τις στήλες, τις μετονομάζουμε και αλλάζουμε τον τύπο δεδομένων τους (Casting)
weather_silver_df = weather_silver_df.select(
    to_date(col("latitude")).alias("weather_date"),
    col("longitude").cast("double").alias("avg_temp"),
    col("elevation").cast("double").alias("precipitation")
)

# 3. Εμφάνιση του πεντακάθαρου πίνακα καιρού
display(weather_silver_df)

## Gold Layer

In [0]:
# 1. Εκτέλεση του JOIN με βάση την ημερομηνία
final_gold_df = taxis_silver_df.join(
    weather_silver_df, 
    taxis_silver_df["pickup_date"] == weather_silver_df["weather_date"], 
    "inner"
)

# 2. Καταχώρηση του Dataframe ως SQL Table στη μνήμη του Spark
final_gold_df.createOrReplaceTempView("gold_nyc_taxi_weather")
display(final_gold_df.limit(10))

## SQL Exploration

In [0]:
%sql
SELECT 
    CASE 
        WHEN precipitation = 0 THEN 'Όχι Βροχή (Στεγνός Καιρός)'
        WHEN precipitation > 0 AND precipitation <= 2 THEN 'Ψιχάλα / Ελαφριά Βροχή'
        ELSE 'Καταιγίδα / Έντονη Βροχή'
    END as weather_condition,
    COUNT(*) as total_rides,
    ROUND(AVG(trip_distance), 2) as avg_distance,
    ROUND(AVG(tip_amount), 2) as avg_tip_amount,
    ROUND(AVG(total_amount), 2) as avg_total_paid
FROM gold_nyc_taxi_weather
GROUP BY 1
ORDER BY avg_tip_amount DESC;

Databricks visualization. Run in Databricks to view.

## Save Delta Table

In [0]:
# Αποθήκευση του τελικού πίνακα μόνιμα στο Catalog σου
final_gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_taxi_weather_analytics")

## END OF PRACTICE